# Pair-level coordination in V1 (BTBR vs C57BL/6J) from `resp_by_subject_angle`

Run this after the cell that builds `pop_data`. It uses only `pop_data[geno]['resp_by_subject_angle'][sid][angle]`, arrays of shape `(N_neurons, T_resp, n_trials)`. No new loaders.

Time bins, not trials, are the unit of estimation (each animal contributes about 8 angles × T_resp × n_trials bins), and every comparison is made **per animal**.

**Assumed, not yet checked against the data**
- Every animal has the same neurons and the same `T_resp` across angles (checked below; the cell fails loudly if not).
- The arrays contain only responsive neurons, so all pair sets are drawn from neurons selected on their evoked response.

## What is in `pop_data`?

Label the fields for one animal before trusting anything else.

In [ ]:
import numpy as np

GENOS = sorted(pop_data.keys(), key=lambda g: 'BTBR' in str(g).upper())   # control first, BTBR second
assert len(GENOS) == 2, f"expected 2 genotype keys, found {GENOS}"
G_REF, G_EXP = GENOS
COLORS = {g: ('#2E8B57' if 'BTBR' in str(g).upper() else 'black') for g in GENOS}

RBSA = {g: pop_data[g]['resp_by_subject_angle'] for g in GENOS}
GENO_OF = {sid: g for g in GENOS for sid in RBSA[g]}
assert len(GENO_OF) == sum(len(RBSA[g]) for g in GENOS), "animal ids repeat across genotypes"

print("animals per genotype:", {g: len(RBSA[g]) for g in GENOS})
g0 = G_REF; s0 = next(iter(RBSA[g0]))
print(f"\nexample animal : {s0} ({g0!r})")
for a in sorted(RBSA[g0][s0]):
    print(f"  angle {a}: shape {RBSA[g0][s0][a].shape}   (neurons, frames, trials)")

NameError: name 'pop_data' is not defined

## Which quantities does each animal contribute?

For every animal:
- **Residuals.** For each angle, the mean time course across trials is subtracted, so stimulus-locked drive is not counted as coordination. Angles are then pooled along the time-bin axis.
- **Residual correlation** between every pair of neurons over those bins. This is a time-bin measure, not the trial-level noise correlation of Phase 1a, so the numbers will not match.
- **Signal correlation**: correlation of two neurons' tuning curves across angles, computed **across halves of the trials** (odd trials for one neuron, even for the other, both directions averaged) so shared trial noise cannot inflate it. It is attenuated with few trials but not biased upward.
- **Peak evoked amplitude** per neuron, to check that amplitude does not drive the results.

In [ ]:
def corr_matrix(X):
    Xc = X - X.mean(axis=1, keepdims=True)
    sd = Xc.std(axis=1)
    ok = sd > 0
    Z = np.zeros_like(Xc); Z[ok] = Xc[ok] / sd[ok, None]
    C = Z @ Z.T / X.shape[1]
    C[~ok, :] = np.nan; C[:, ~ok] = np.nan
    np.fill_diagonal(C, np.nan)
    return C


def animal_measures(sid, d):
    angs = sorted(d)
    N, T = d[angs[0]].shape[:2]
    for a in angs:
        if d[a].shape[0] != N or d[a].shape[1] != T:
            raise ValueError(f"{sid}: angle {a} has shape {d[a].shape}, angle {angs[0]} has {d[angs[0]].shape}. "
                             f"Neuron sets or T_resp differ across angles; pair analyses need one neuron set per animal.")
        if not np.isfinite(d[a]).all():
            raise ValueError(f"{sid}: angle {a} contains NaN/inf")
        if d[a].shape[2] < 2:
            raise ValueError(f"{sid}: angle {a} has a single trial")
    resid = np.concatenate([(d[a] - d[a].mean(axis=2, keepdims=True)).reshape(N, -1) for a in angs], axis=1)

    resp = {a: d[a].mean(axis=1) for a in angs}                        # (N, trials) per angle
    tun_odd = np.stack([resp[a][:, 0::2].mean(axis=1) for a in angs], axis=1)
    tun_even = np.stack([resp[a][:, 1::2].mean(axis=1) for a in angs], axis=1)
    tuning = np.stack([resp[a].mean(axis=1) for a in angs], axis=1)     # (N, angles)

    def z(M):
        M = M - M.mean(axis=1, keepdims=True); s = M.std(axis=1)
        return M / np.where(s > 0, s, np.nan)[:, None]
    ZA, ZB = z(tun_odd), z(tun_even)
    sc = (ZA @ ZB.T + ZB @ ZA.T) / (2 * len(angs))
    np.fill_diagonal(sc, np.nan)

    peak = np.clip(tuning.max(axis=1), 1e-6, None)
    return dict(resid=resid, sc=sc, rc=corr_matrix(resid), amp=np.sqrt(peak[:, None] * peak[None, :]),
                n_neurons=N, n_bins=resid.shape[1])


M = {sid: animal_measures(sid, RBSA[GENO_OF[sid]][sid]) for sid in GENO_OF}

# Scratch: one row per animal
print(f"{'animal':<14}{'genotype':<10}{'neurons':>8}{'time bins':>11}{'pairs':>8}")
for sid, m in M.items():
    print(f"{str(sid):<14}{str(GENO_OF[sid]):<10}{m['n_neurons']:>8}{m['n_bins']:>11}{m['n_neurons']*(m['n_neurons']-1)//2:>8}")

## How do two genotypes with ~10 animals get compared?

One number per animal (mean, slope or rank correlation over that animal's pairs), then:
- an exact permutation test over genotype labels;
- a hierarchical bootstrap (animals, then units within animal; Saravanan et al. 2020), 95% interval for `experimental − control`.

If the two disagree, read the permutation p first: with few animals in a group the bootstrap interval tends to be too narrow.

In [ ]:
import itertools
import matplotlib.pyplot as plt

N_BOOT, SEED = 5000, 0
RESULTS = []


def exact_perm_p(a, b):
    x = np.concatenate([a, b]); n = len(a); obs = a.mean() - b.mean(); diffs = []
    for idx in itertools.combinations(range(len(x)), n):
        m = np.zeros(len(x), bool); m[list(idx)] = True
        diffs.append(x[m].mean() - x[~m].mean())
    diffs = np.array(diffs)
    return float((np.abs(diffs) >= abs(obs) - 1e-12).mean()), 2.0 / len(diffs)


def hier_boot_diff(units):
    rng = np.random.default_rng(SEED)
    by_g = {g: [v for s, v in units.items() if GENO_OF[s] == g] for g in GENOS}
    d = np.empty(N_BOOT)
    for b in range(N_BOOT):
        gm = {}
        for g, lst in by_g.items():
            pick = rng.integers(0, len(lst), len(lst))
            gm[g] = np.mean([rng.choice(lst[i], len(lst[i]), replace=True).mean() for i in pick])
        d[b] = gm[G_EXP] - gm[G_REF]
    return np.percentile(d, [2.5, 97.5])


def compare(metric, units):
    """units: animal -> 1D array of unit values (length 1 for animal-level scalars)."""
    units = {s: np.asarray(v, float)[np.isfinite(v)] for s, v in units.items()}
    units = {s: v for s, v in units.items() if len(v)}
    mean = {s: v.mean() for s, v in units.items()}
    ref = np.array([mean[s] for s in units if GENO_OF[s] == G_REF])
    exp = np.array([mean[s] for s in units if GENO_OF[s] == G_EXP])
    if len(ref) == 0 or len(exp) == 0:
        print(f"{metric:<44} not compared: n = {len(ref)} vs {len(exp)} animals with data"); return
    row = dict(metric=metric, n_ref=len(ref), n_exp=len(exp), ref=ref.mean(), exp=exp.mean(), diff=exp.mean() - ref.mean(),
               ci_lo=np.nan, ci_hi=np.nan, p_perm=np.nan, p_min=np.nan)
    if len(ref) >= 2 and len(exp) >= 2:
        row['p_perm'], row['p_min'] = exact_perm_p(exp, ref)
    if len(ref) >= 3 and len(exp) >= 3:
        row['ci_lo'], row['ci_hi'] = hier_boot_diff(units)
    RESULTS.append(row)
    ci = f"[{row['ci_lo']:.3f}, {row['ci_hi']:.3f}]" if np.isfinite(row['ci_lo']) else "n/a"
    print(f"{metric:<44} {G_REF} {row['ref']:.3f} (n={len(ref)}) | {G_EXP} {row['exp']:.3f} (n={len(exp)}) | "
          f"diff {row['diff']:+.3f} | 95% CI {ci} | p_perm {row['p_perm']:.3f} (min {row['p_min']:.3f})")


def group_box(ax, per_animal, ylabel=''):
    rng = np.random.default_rng(1)
    for k, g in enumerate(GENOS):
        v = np.array([x for s, x in per_animal.items() if GENO_OF[s] == g], float); v = v[np.isfinite(v)]
        c = COLORS[g]
        if len(v) >= 2:
            ax.boxplot(v, positions=[k], widths=0.5, showfliers=False, boxprops=dict(color=c), whiskerprops=dict(color=c),
                       capprops=dict(color=c), medianprops=dict(color=c))
        ax.scatter(k + rng.uniform(-0.12, 0.12, len(v)), v, s=28, color=c, zorder=3)
    n = {g: int(np.sum([np.isfinite(x) for s, x in per_animal.items() if GENO_OF[s] == g])) for g in GENOS}
    ax.set_xticks(range(2)); ax.set_xticklabels([f"{g}\n(n={n[g]})" for g in GENOS]); ax.set_ylabel(ylabel)
    for s in ('top', 'right'):
        ax.spines[s].set_visible(False)


def binned_by_quantile(xy, n_bins=5, min_pairs=30):
    """xy: animal -> (x, y). Pooled-quantile bins on x; per-animal mean of y per bin (NaN if < min_pairs)."""
    xs = np.concatenate([x for x, _ in xy.values()])
    edges = np.quantile(xs, np.linspace(0, 1, n_bins + 1))
    centers = np.array([np.median(xs[(xs >= edges[k]) & (xs <= edges[k + 1])]) for k in range(n_bins)])
    tab = {}
    for s, (x, y) in xy.items():
        b = np.clip(np.digitize(x, edges[1:-1]), 0, n_bins - 1)
        tab[s] = np.array([y[b == k].mean() if (b == k).sum() >= min_pairs else np.nan for k in range(n_bins)])
    return centers, tab


def panel_binned(ax, centers, tab, xlabel, ylabel):
    for g in GENOS:
        Mg = np.array([tab[s] for s in tab if GENO_OF[s] == g])
        if Mg.size == 0:
            continue
        for row in Mg:
            ax.plot(centers, row, color=COLORS[g], lw=0.6, alpha=0.4)
        ok = np.isfinite(Mg); n = ok.sum(axis=0)
        m = np.where(n > 0, np.where(ok, Mg, 0).sum(axis=0) / np.maximum(n, 1), np.nan)
        se = np.where(n > 1, np.sqrt(np.where(ok, (Mg - m) ** 2, 0).sum(axis=0) / np.maximum(n - 1, 1)) / np.sqrt(np.maximum(n, 1)), 0)
        ax.errorbar(centers, m, yerr=se, color=COLORS[g], lw=2, marker='o', ms=4, label=f"{g} (n={Mg.shape[0]})")
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel); ax.legend(frameon=False)
    for s in ('top', 'right'):
        ax.spines[s].set_visible(False)

## Does coordination respect tuning similarity?

In V1, neurons with similar tuning tend to co-fluctuate more (Cohen & Kohn 2011). A reduced OSI says the tuning curves are broader; it does not say whether the circuit still coordinates like with like, and that is what this section asks.

Per animal: slope and Spearman rho of residual correlation against signal correlation, plus the slope divided by the mean residual correlation. The slope and rho both scale with the overall level of correlation (a group with more shared variance also has a less noisy relationship), so the three rows have to be read together.

In [ ]:
from scipy.stats import spearmanr

MIN_PAIRS_FIT, MIN_PAIRS_BIN = 100, 30
iu = lambda m: m[np.triu_indices_from(m, k=1)]

xy_sc, slope, rel, rho = {}, {}, {}, {}
for sid, m in M.items():
    x, y = iu(m['sc']), iu(m['rc'])
    ok = np.isfinite(x) & np.isfinite(y)
    xy_sc[sid] = (x[ok], y[ok])
    slope[sid] = rel[sid] = rho[sid] = np.nan
    if ok.sum() >= MIN_PAIRS_FIT:
        slope[sid] = np.polyfit(x[ok], y[ok], 1)[0]
        rho[sid] = spearmanr(x[ok], y[ok])[0]
        rel[sid] = slope[sid] / y[ok].mean() if y[ok].mean() > 0.01 else np.nan

compare('mean residual correlation', {s: np.array([np.nanmean(iu(m['rc']))]) for s, m in M.items()})
compare('like-to-like slope', {s: np.array([v]) for s, v in slope.items()})
compare('like-to-like slope / mean corr.', {s: np.array([v]) for s, v in rel.items()})
compare('like-to-like rho', {s: np.array([v]) for s, v in rho.items()})
compare('mean signal correlation', {s: np.array([np.nanmean(iu(m['sc']))]) for s, m in M.items()})

centers_sc, tab_sc = binned_by_quantile(xy_sc, 5, MIN_PAIRS_BIN)
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
panel_binned(axes[0], centers_sc, tab_sc, 'Signal correlation (cross-split)', 'Residual correlation')
group_box(axes[1], slope, 'Slope: residual vs signal corr.')
group_box(axes[2], rel, 'Slope / mean residual corr.')
plt.tight_layout(); plt.show()

Open: pair sets come from neurons selected as responsive, so tuning-related structure is partly built in by the selection. The comparison between genotypes is still fair only if the selection criterion is the same in both.

## Is the difference explained by how strongly the neurons respond?

Correlations grow with signal-to-noise. If pair amplitude differs between genotypes, or the residual correlation changes with amplitude, the previous section could be an amplitude effect. Pairs are binned by the geometric mean of their two neurons' peak evoked amplitude; if the genotype curves overlap, amplitude is not what separates them.

In [ ]:
xy_amp = {}
for sid, m in M.items():
    x, y = iu(m['amp']), iu(m['rc'])
    ok = np.isfinite(x) & np.isfinite(y)
    xy_amp[sid] = (x[ok], y[ok])

compare('mean pair amplitude', {s: np.array([np.mean(v[0])]) for s, v in xy_amp.items()})
centers_amp, tab_amp = binned_by_quantile(xy_amp, 5, MIN_PAIRS_BIN)
fig, axes = plt.subplots(1, 2, figsize=(7, 3))
panel_binned(axes[0], centers_amp, tab_amp, 'Pair peak amplitude (geometric mean)', 'Residual correlation')
group_box(axes[1], {s: np.mean(v[0]) for s, v in xy_amp.items()}, 'Mean pair peak amplitude')
plt.tight_layout(); plt.show()

Open: peak amplitude is a proxy. Event rate would be the better control, and it needs the continuous trace.

## How unevenly is each neuron coupled to the rest of the population?

Population coupling (Okun et al. 2015): correlation between a neuron's residual activity and the summed residual activity of all the others. It condenses the pairwise structure into one number per neuron and connects directly to the population level. It depends on how many neurons form "the rest", so it is computed on random subsets of a **fixed size** `POP_N` in every animal.

In [ ]:
N_SUBSAMPLE = 200
POP_N = int(min(min(m['n_neurons'] for m in M.values()), 60))
if POP_N < 10:
    raise ValueError(f"Smallest field has {POP_N} neurons; population coupling needs at least 10.")
print(f"POP_N = {POP_N} (smallest field, capped at 60)")


def population_coupling(X, n_sub, n_rep, rng):
    Xc = X - X.mean(axis=1, keepdims=True); sd = Xc.std(axis=1)
    Z = Xc[sd > 0] / sd[sd > 0, None]
    N = Z.shape[0]
    acc, cnt, means, sds = np.zeros(N), np.zeros(N), [], []
    for _ in range(n_rep):
        idx = rng.choice(N, n_sub, replace=False)
        Zs = Z[idx]; tot = Zs.sum(axis=0)
        c = (Zs * tot).mean(axis=1)
        r = (c - 1) / np.sqrt(np.maximum(tot.var() - 2 * c + 1, 1e-12))
        acc[idx] += r; cnt[idx] += 1; means.append(r.mean()); sds.append(r.std())
    with np.errstate(invalid='ignore'):
        return acc / np.where(cnt > 0, cnt, np.nan), float(np.mean(means)), float(np.mean(sds))


rng = np.random.default_rng(SEED)
coup_units, coup_mean, coup_sd = {}, {}, {}
for sid, m in M.items():
    u, mu, sd_ = population_coupling(m['resid'], POP_N, N_SUBSAMPLE, rng)
    coup_units[sid], coup_mean[sid], coup_sd[sid] = u[np.isfinite(u)], mu, sd_

compare('population coupling, mean (fixed N)', coup_units)
compare('population coupling, SD across neurons', {s: np.array([v]) for s, v in coup_sd.items()})

fig, axes = plt.subplots(1, 2, figsize=(6.5, 3))
group_box(axes[0], coup_mean, 'Mean population coupling')
group_box(axes[1], coup_sd, 'SD of coupling across neurons')
plt.tight_layout(); plt.show()

## Summary

In [ ]:
import pandas as pd
print(pd.DataFrame(RESULTS).set_index('metric').round(3).to_string())

### Unresolved

- Coordination in the gray intervals (STTC, gray vs stimulus, gray–tuning alignment) is not here: it needs the gray frames of the continuous trace, which `resp_by_subject_angle` does not hold.
- Residual correlations over time bins are not the trial-level noise correlations of Phase 1a; the two should not be plotted as if they were the same quantity.
- Calcium frames are autocorrelated, so the effective number of time bins is smaller than `n_bins`. This does not affect the per-animal comparisons but it does mean single pair estimates are less precise than they look.
- Group sizes are small: report intervals and exact p-values as they are.